# SCHISMlab: análisis de una simulación

Cambia solo la celda de **parámetros** y ejecuta el cuaderno entero (Run > Run All Cells).

In [ ]:
# ---------------- PARÁMETROS ----------------
CASO = "../cases/teaching/ekman"      # carpeta de ejecución del caso (donde está outputs/)
VARIABLE = "elevation"                # elevation, depthAverageVel, horizontalVel, temperature, salinity, ...
PUNTO = (2.5, 39.4)                   # lon, lat (o x, y en metros si el dominio es cartesiano)
SECCION = ((2.0, 39.4), (3.0, 39.4))  # extremos de la sección vertical (solo casos 3D)

In [ ]:
import matplotlib.pyplot as plt
from schismlab.analysis import open_run, plot_map, plot_timeseries, plot_profile, plot_section, animate
from schismlab.case import Case
from schismlab.setup import RunSetup
ds = open_run(CASO)
print("variables:", [v for v in ds.data_vars if "time" in ds[v].dims])
print("instantes:", ds.time.size, "desde", ds.time.values[0], "hasta", ds.time.values[-1])

## Mapa en el último instante

In [ ]:
plot_map(ds, VARIABLE, time=-1, vectors=True);

## Serie temporal en un punto

In [ ]:
plot_timeseries(ds, VARIABLE, *PUNTO);

## Perfil vertical y sección (solo si la simulación es 3D)

In [ ]:
import glob, yaml
yml = glob.glob(CASO.rstrip("/") + ".yaml") or glob.glob(CASO.rstrip("/") + "/../" + CASO.rstrip("/").split("/")[-1] + ".yaml")
if "layer" in ds.dims and yml:
    rs = RunSetup(Case(yml[0]), CASO); rs.m = None; vg = rs.vertical.__func__.__globals__["VGrid"]  # placeholder
    from schismlab.forcing import VGrid
    vc = Case(yml[0]).cfg.get("vertical", {})
    vg = VGrid.sigma(int(vc.get("nvrt", 20)), float(vc.get("h_c", 10)), float(vc.get("theta_b", 0)), float(vc.get("theta_f", 4)))
    fig, ax = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={"width_ratios": [1, 3]})
    plot_profile(ds, "horizontalVel", *PUNTO, vg, ax=ax[0])
    plot_section(ds, "temperature" if "temperature" in ds else "horizontalVelX", SECCION[0], SECCION[1], vg, ax=ax[1])
else:
    print("simulación 2D o sin YAML: no hay perfiles ni secciones")

## Animación

In [ ]:
p = animate(ds, VARIABLE, f"{CASO}/animacion_{VARIABLE}.mp4", step=1)
from IPython.display import Video, Image
Video(str(p), embed=True) if p.suffix == ".mp4" else Image(str(p))